In [8]:
import os
import math
from functools import reduce
from operator import add
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import NumericType
from pyspark.sql.functions import (
    col, when, count,
    to_date, dayofmonth, month, year, date_format, to_timestamp,
    hour as spark_hour, 
    sin, cos, last, first
)
from pyspark.sql import functions as sf
spark = SparkSession.builder \
    .appName("Weather_Preprocessing_Jupyter") \
    .master("local[*]") \
    .config("spark.driver.memory", "24g") \
    .config("spark.executor.memory", "24g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()


In [9]:
def step_1_drop_cols(df: DataFrame) -> DataFrame:
    # 1. Dổi tên cột
    new_columns = [
        'index', 'date', 'hour', 'prcp', 'askq', 'askkmax', 'askkmin', 'bxmt', 
        'tempkk', 'tempds', 'tmax', 'tmin', 'tdsmax', 'tdsmin', 'ttdmax', 'ttdmin', 
        'humi', 'windydirect', 'windygust', 'windyspeed', 'region', 'state', 'station', 
        'station_code', 'latitude', 'longitude', 'elevation'
    ]
    
    if len(df.columns) == len(new_columns):
        df = df.toDF(*new_columns)
    else:
        print(f"CẢNH BÁO: Số lượng cột thực tế ({len(df.columns)}) không khớp ({len(new_columns)}).")
    # 2. Bỏ cột không sài
    cols_to_drop = ['index', 'region', 'state', 'station']
    df = df.drop(*cols_to_drop)
    # 3. Định dạng lại cột hour (2026-03-16 00:00:00 -> 00:00:00)
    df = df.withColumn('hour', date_format(to_timestamp(col('hour')), 'HH:mm:ss'))

    # 4. Xử lý cột date và lọc năm
    if 'date' in df.columns:
        # TẠO CỘT MỚI: Ép kiểu chuỗi về dạng chuẩn DateType và lưu vào 'date_casted'
        df = df.withColumn('date_casted', sf.to_date(sf.col('date')))
        
        # Dùng cột mới 'date_casted' này để bóc tách, cột 'date' gốc vẫn y nguyên
        df = df.withColumn('day', sf.dayofmonth(sf.col('date_casted')))
        df = df.withColumn('month', sf.month(sf.col('date_casted')))
        df = df.withColumn('year', sf.year(sf.col('date_casted')))
        
        # Vì đã tách xong mục đích chính, mình xóa cột tạm này đi cho gọn, giữ lại date gốc
        df = df.drop('date_casted')
    df = df.filter(col('year') >= 2021)
    return df


def step_2_handle_missing(df):
    numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]
    all_cols = df.columns
    
    # 1. Chuyển các dữ liệu lỗi thành Null
    replace_exprs = [
        when(col(c) == -9999, None).otherwise(col(c)).alias(c) if c in numeric_cols else col(c) 
        for c in all_cols
    ]
    df = df.select(*replace_exprs)

    # 2. Xóa những ngày dữ liệu toàn lỗi
    cols_to_check = [
        'prcp', 'askq', 'askkmax', 'askkmin', 'bxmt', 
        'tempkk', 'tempds', 'tmax', 'tmin', 'tdsmax', 
        'tdsmin', 'ttdmax', 'ttdmin', 'humi', 'windydirect', 
        'windygust', 'windyspeed'
    ]
    
    # Chỉ xét những cột có thật trong df
    actual_cols = [c for c in cols_to_check if c in df.columns]

    if actual_cols and "station_code" in df.columns and "date" in df.columns:
        # Đếm số lượng Not Null của từng cột (Gom nhóm theo TRẠM và NGÀY)
        agg_exprs = [count(col(c)).alias(f"cnt_{c}") for c in actual_cols]
        daily_stats = df.groupBy("station_code", "date").agg(*agg_exprs)

        # Cộng dồn tất cả các biến count lại (Nếu = 0 nghĩa là trọn 24h của ngày đó đều NULL)
        total_valid_expr = reduce(add, [col(f"cnt_{c}") for c in actual_cols])

        # Lọc ra danh sách các NGÀY có ít nhất 1 dòng dữ liệu hợp lệ (> 0)
        valid_days = daily_stats.filter(total_valid_expr > 0).select("station_code", "date")

        # Inner Join: Những ngày toàn NULL sẽ bị loại bỏ sạch sẽ!
        df = df.join(valid_days, on=["station_code", "date"], how="inner")

    # 3. Điền dữ liệu khuyết bằng dữ liệu gần nhất
    # Sắp xếp theo ngày 
    window_ffill = Window.partitionBy("station_code").orderBy("date", "hour") \
                         .rowsBetween(Window.unboundedPreceding, Window.currentRow)
                         
    window_bfill = Window.partitionBy("station_code").orderBy("date", "hour") \
                         .rowsBetween(Window.currentRow, Window.unboundedFollowing)
    
    # Kéo giá trị hợp lệ từ quá khứ xuống
    ffill_exprs = [
        last(col(c), ignorenulls=True).over(window_ffill).alias(c) if c in numeric_cols else col(c) 
        for c in all_cols
    ]
    df = df.select(*ffill_exprs)

    # Đẩy giá trị hợp lệ từ tương lai lên nếu không có giá trị ngày quá khứ
    bfill_exprs = [
        first(col(c), ignorenulls=True).over(window_bfill).alias(c) if c in numeric_cols else col(c) 
        for c in all_cols
    ]
    df = df.select(*bfill_exprs)

    return df

def step_3_rain_label(df: DataFrame) -> DataFrame:
    # Gán nhãn
    if 'prcp' in df.columns:
        df = df.withColumn('rain_label', when(col('prcp') > 0, 1).otherwise(0))
    return df

def step_4_datetime_features(df: DataFrame) -> DataFrame:
    # 1. Tính Sin/Cos cho Tháng (Chu kỳ 12 tháng)
    if 'month' in df.columns:
        # Công thức: sin(month * 2 * pi / 12)
        df = df.withColumn('month_sin', sin(col('month') * (2 * math.pi / 12)))
        df = df.withColumn('month_cos', cos(col('month') * (2 * math.pi / 12)))

    # 2. Tính Sin/Cos cho Giờ (Chu kỳ 24 giờ)
    if 'hour' in df.columns:
        # Lấy số giờ từ định dạng HH:mm:ss đã làm ở Bước 1
        df = df.withColumn('_temp_hour', spark_hour(col('hour')).cast('double'))
        
        df = df.withColumn('hour_sin', sin(col('_temp_hour') * (2 * math.pi / 24)))
        df = df.withColumn('hour_cos', cos(col('_temp_hour') * (2 * math.pi / 24)))
        
        # Xóa cột tạm
        df = df.drop('_temp_hour')

    return df


def step_5_compute_iqr(
    df: DataFrame,
    columns: list[str] = None,
    relative_error: float = 0.001,
    factor: float = 1.5
) -> DataFrame:
    """
    Xử lý giá trị ngoại lai (Outliers) bằng phương pháp IQR trên TOÀN BỘ dữ liệu.
    Thay thế các giá trị nằm ngoài ngưỡng [Q1 - 1.5*IQR, Q3 + 1.5*IQR] bằng Median (Q2).
    """

    # 1. Tự động xác định danh sách các cột số cần xử lý (Weather metrics)
    if columns is None:
        # Loại trừ các cột định danh, tọa độ và các cột thời gian đã biến đổi
        exclude_cols = {
            'station_code', 'latitude', 'longitude', 'elevation', 
            'rain_label', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos',
            'date', 'hour', 'year', 'month', 'day'
        }
        columns = [
            f.name for f in df.schema.fields 
            if isinstance(f.dataType, NumericType) and f.name not in exclude_cols
        ]

    print(f"-> Đang tính toán IQR cho {len(columns)} cột trên toàn bộ dữ liệu...")
    
    # 2. Tính toán các phân vị (Quantiles) xấp xỉ
    # Chạy trên df gốc (100% dữ liệu) để lấy Q1 (25%), Median (50%), Q3 (75%)
    # relative_error = 0.001 giúp cân bằng độ chính xác khi quét tập dữ liệu lớn
    quantiles_list = df.approxQuantile(columns, [0.25, 0.5, 0.75], relative_error)

    # 3. Duyệt qua kết quả tính toán để xác định ngưỡng cho từng cột
    col_dict = {}
    for i, col_name in enumerate(columns):
        q1, q2, q3 = quantiles_list[i]
        iqr = q3 - q1
        
        lower_bound = q1 - factor * iqr
        upper_bound = q3 + factor * iqr
        
        # Lưu lại ngưỡng và giá trị trung vị (Median) để thay thế
        col_dict[col_name] = (lower_bound, upper_bound, q2)

    # 4. Xây dựng danh sách biểu thức thay thế (Vectorized Transformation)
    replace_exprs = []
    for c in df.columns:
        if c in columns:
            lower, upper, q2 = col_dict[c]
            # Nếu giá trị ngoài khoảng [lower, upper] thì thay bằng Median (q2)
            expr = sf.when((sf.col(c) < lower) | (sf.col(c) > upper), sf.lit(q2)) \
                     .otherwise(sf.col(c)) \
                     .alias(c)
            replace_exprs.append(expr)
        else:
            # Giữ nguyên các cột không thuộc diện xử lý IQR
            replace_exprs.append(sf.col(c))

    # 5. Thực thi biến đổi toàn bộ DataFrame trong 1 lần select duy nhất
    # Việc gom nhóm này giúp Spark tối ưu hóa kế hoạch thực thi (DAG)
    return df.select(*replace_exprs)


def step_6_add_prep(df: DataFrame) -> DataFrame:
    # 1. Tạo Đặc trưng Độ trễ (Lag) cho 17 chỉ số với tiền tố 'pre_'
    lag_columns = [
        'prcp', 'askq', 'askkmax', 'askkmin', 'bxmt', 'tempkk', 'tempds', 
        'tmax', 'tmin', 'tdsmax', 'tdsmin', 'ttdmax', 'ttdmin', 'humi', 
        'windydirect', 'windygust', 'windyspeed'
    ]

    # Sắp xếp trực tiếp bằng cột 'date' và 'hour' sẵn có
    if 'date' in df.columns and 'hour' in df.columns:
        window_spec = Window.partitionBy("station_code").orderBy("date", "hour")
        
        for col_name in lag_columns:
            if col_name in df.columns:
                pre_col_name = f"pre_{col_name}"  # Tạo tên cột mới có chữ pre_
                
                # Kéo dữ liệu của giờ trước
                df = df.withColumn(pre_col_name, sf.lag(col_name, 1).over(window_spec))
                
                # Lấp giá trị Null của dòng đầu tiên bằng chính giá trị dòng đó
                df = df.withColumn(pre_col_name, sf.coalesce(sf.col(pre_col_name), sf.col(col_name)))

    # 2. Dọn dẹp các cột thời gian gốc (Tránh học vẹt)
    cols_to_drop_final = ['date', 'hour', 'year', 'day', 'month']
    existing_cols = [c for c in cols_to_drop_final if c in df.columns]
    
    df = df.drop(*existing_cols)

    return df

def run_preprocessing_pipeline(df: DataFrame) -> DataFrame:
    print("-> Chạy Bước 1...")
    df1 = step_1_drop_cols(df)

    print("-> Chạy Bước 2...")
    df2 = step_2_handle_missing(df1)

    print(df.count())
    
    df2.cache()

    print("-> Chạy Bước 3...")
    df3 = step_3_rain_label(df2)

    print("-> Chạy Bước 4...")
    df4 = step_4_datetime_features(df3)

    print("-> Chạy Bước 5...")
    df5 = step_5_compute_iqr(df4)

    print("-> Chạy Bước 6...")
    df_final = step_6_add_prep(df5)

    df2.unpersist()

    return df_final

In [10]:
input_file = 'data.parquet'
print(f"Đang đọc dữ liệu từ '{input_file}'...")
df = spark.read.parquet(input_file)
print("\n--- BẮT ĐẦU LUỒNG TIỀN XỬ LÝ BƯỚC 1 TỚI 6 ---")
df_processed_1 = run_preprocessing_pipeline(df)


Đang đọc dữ liệu từ 'data.parquet'...

--- BẮT ĐẦU LUỒNG TIỀN XỬ LÝ BƯỚC 1 TỚI 6 ---
-> Chạy Bước 1...
-> Chạy Bước 2...
16260936
-> Chạy Bước 3...
-> Chạy Bước 4...
-> Chạy Bước 5...
-> Đang tính toán IQR cho 17 cột trên toàn bộ dữ liệu...


-> Chạy Bước 6...


In [11]:
output_parquet_path = "data_processed_final.parquet"

print(f"Đang xuất dữ liệu ra định dạng Parquet tại: '{output_parquet_path}'...")

df_processed_1.write.mode("overwrite").parquet(output_parquet_path)

print("Đã xuất file Parquet thành công!")

file_name = "danh_sach_cot.txt"
with open(file_name, "w", encoding="utf-8") as f:
    f.write(f"--- TỔNG CỘNG: {len(df_processed_1.columns)} CỘT ---\n")
    for i, col_name in enumerate(df_processed_1.columns, 1):
        f.write(f"{i}. {col_name}\n")

print(f"Đã cập nhật danh sách {len(df_processed_1.columns)} cột vào file '{file_name}'.")

Đang xuất dữ liệu ra định dạng Parquet tại: 'data_processed_final.parquet'...


Đã xuất file Parquet thành công!
Đã cập nhật danh sách 43 cột vào file 'danh_sach_cot.txt'.
